<big><big><big>001 - Query Fritz</big></big></big><br>


what does this notebook do:
- shows (for the most part) how I made my giant dataframe `obj_spectra_info.csv`
    - part 1: object classifications
    - part 2: additional object information (RA, Dec, host galaxy, etc)
    - part 3: object spectra
    

<br>

> <u>note about querying from Fritz:</u>
> - my <i>actual</i> token (not included in `fritz_query.py`) does not grant me to access to all the sources in the dataset. if you're querying using `fritz_query.query_classification` or `fritz_query.query_info` for the same objects in my dataset, you should not be getting any objects that show up in the <i>fail_req</i>, <i>fail_data</i> lists. if you do, either you were rate limited (unlikely as there is a buffer between each request) or you don't have access to that object.
> - solution: steal theophile's token (also not included in `fritz_query.py`)


In [1]:
# packages
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# query.py
from src_dataloader import fritz_query

In [2]:
# example query
obj_id_list = ['ZTF17aaaobyl', 'ZTF18aatfxtd', 'ZTF18aabdajx' ]
obj_class, fail_req, fail_data = fritz_query.query_classification(obj_id_list)

Processing Objects:   0%|          | 0/3 [00:00<?, ?it/s]

ZTF17aaaobyl classified with prob: 1.0 as Cataclysmic
ZTF18aatfxtd classified with prob: 0.76 as Ia
ZTF18aabdajx classified with prob: 1.0 as Tidal Disruption Event


<big>part 1: query object classifications</big>

`fritz_query.query_classifications(obj_id)`
- **Parameters**:
    - `obj_id` : list of ZTF IDs
- **Functionality**:
  - finds most recent classification with probablity >= 0.6 (and not classified by machine learning algo)
- **Returns**:
    - `obj_class`: list of object IDs and their classifications that meet requirements
    - `fail_req`: list of object IDs that failed classificaiton requirements (low prob, ML classified)
    - `fail_data`: list of object IDs that query failed for (rate limit exceeded, etc)


In [3]:
obj_class

[('ZTF17aaaobyl', 'Cataclysmic'),
 ('ZTF18aatfxtd', 'Ia'),
 ('ZTF18aabdajx', 'Tidal Disruption Event')]

In [4]:
# create a dataframe
obj_class_df = pd.DataFrame(obj_class, columns=['obj_id', 'type'])
obj_class_df

,obj_id,type
0,ZTF17aaaobyl,Cataclysmic
1,ZTF18aatfxtd,Ia
2,ZTF18aabdajx,Tidal Disruption Event


<big>part 2: query object info</big>

`fritz_query.query_info(obj_id)`
- **Parameters**:
    - `obj_id` : list of ZTF IDs
- **Functionality**:
  - finds most recent classification with probablity >= 0.6 (and not classified by machine learning algo)
- **Returns**:
    - `obj_info`: list wit object ID, RA, Dec, Transient Name Survey (TNS) ID, host galaxy name (some don't include this), gal lat, gal lon, luminosity distance
    - `no_info`: list of object IDs missing additional information
    - `fail_data`: list of object IDs that query failed for (rate limit exceeded, etc)


<br><br>

who cares about this additional information anyways?
- we will RA, Dec to query for spectra from other databases/surveys (SDSS, DESI)
- host galaxy information used for host galaxy spectra (see later notebook)


In [5]:
obj_info, no_info, fail_data = fritz_query.query_info(obj_id_list)

Processing Objects:   0%|          | 0/3 [00:00<?, ?it/s]

In [6]:
obj_info

[('ZTF17aaaobyl',
  '2018fjo',
  93.8348102,
  48.4867887,
  None,
  14.41325716775719,
  165.45291430926642,
  None),
 ('ZTF18aatfxtd',
  'AT 2024erf',
  175.4927526,
  51.4873192,
  ['WISEA J114158.27+512914.7'],
  62.33968788244336,
  146.53333478370163,
  217.99753085006677),
 ('ZTF18aabdajx',
  None,
  185.1878936,
  49.5513609,
  ['WISEA J122045.05+493304.7'],
  66.81294430271655,
  135.63876588403534,
  128.8067184045032)]

In [7]:
# create a dataframe
obj_info_df = pd.DataFrame(obj_info, columns=['obj_id', 'TNS_id', 'RA', 'Dec', 'host gal', 'gal lat', 'gal lon', 'luminosity distance'])
obj_info_df

,obj_id,TNS_id,RA,Dec,host gal,gal lat,gal lon,luminosity distance
0,ZTF17aaaobyl,2018fjo,93.834810,48.486789,None,14.413257,165.452914,NaN
1,ZTF18aatfxtd,AT 2024erf,175.492753,51.487319,[WISEA J114158.27+512914.7],62.339688,146.533335,217.997531
2,ZTF18aabdajx,None,185.187894,49.551361,[WISEA J122045.05+493304.7],66.812944,135.638766,128.806718


<big>part 3: query spectra</big>

`fritz_query.query_spectra(obj_id)`
- **Parameters**:
    - `obj_id` : list of ZTF IDs
- **Functionality**:
  - finds avaliable spectra
- **Returns**:
    - `data_spectra`: most recent spectra for each object (some objects have multiple spectra)
      - includes: object ID, wavelength, flux, observed date, observed date (MJD), instrument name, telescope name, data length (number of spectra in Fritz) 
    - `no_spectra`: list of object IDs without spectra
    - `fail_data`: list of object IDs that spectra query failed for



In [8]:
data_spectra, no_spectra, fail_data = fritz_query.query_spectra(obj_id_list)

Processing Objects:   0%|          | 0/3 [00:00<?, ?it/s]

ZTF17aaaobyl 1 spectra
ZTF18aatfxtd 1 spectra
ZTF18aabdajx 14 spectra


In [14]:
data_spectra[:1]

[('ZTF17aaaobyl',
  3776.7,
  3.432e-15,
  '2018-08-26T00:00:00',
  58356.0,
  'SEDM',
  'Palomar 1.5m',
  1)]

In [10]:
obj_spectra_df = pd.DataFrame(data_spectra, columns=['obj_id', 'wavelength', 'flux', 'observed_at', 'observed_at_mjd', 'instrument_name', 'telescope_name', 'data_length'])
obj_spectra_df

,obj_id,wavelength,flux,observed_at,observed_at_mjd,instrument_name,telescope_name,data_length
0,ZTF17aaaobyl,3776.7,3.432000e-15,2018-08-26T00:00:00,58356.000000,SEDM,Palomar 1.5m,1
1,ZTF17aaaobyl,3802.3,3.256000e-15,2018-08-26T00:00:00,58356.000000,SEDM,Palomar 1.5m,1
2,ZTF17aaaobyl,3827.9,3.104000e-15,2018-08-26T00:00:00,58356.000000,SEDM,Palomar 1.5m,1
3,ZTF17aaaobyl,3853.4,2.948000e-15,2018-08-26T00:00:00,58356.000000,SEDM,Palomar 1.5m,1
4,ZTF17aaaobyl,3879.0,3.319000e-15,2018-08-26T00:00:00,58356.000000,SEDM,Palomar 1.5m,1
...,...,...,...,...,...,...,...,...
637,ZTF18aabdajx,9121.0,6.086000e-16,2022-02-17T06:57:37,59627.290012,SEDM,Palomar 1.5m,14
638,ZTF18aabdajx,9146.6,6.005000e-16,2022-02-17T06:57:37,59627.290012,SEDM,Palomar 1.5m,14
639,ZTF18aabdajx,9172.1,5.786000e-16,2022-02-17T06:57:37,59627.290012,SEDM,Palomar 1.5m,14
640,ZTF18aabdajx,9197.7,5.388000e-16,2022-02-17T06:57:37,59627.290012,SEDM,Palomar 1.5m,14


In [11]:
# then you can use the below function to save each object's indivdual spectra to .csv in their own folder

#fritz_query.save_obj_spectra_to_obj_folders(obj_spectra_df, obj_id_list, base_path)

optionally, you can get the difference between the date of photometry and the date of spectra 



<big>part 3.1: get photometry, spectra dates</big>

`fritz_query.SpectraAnalysis.get_date_difference_df(obj_spectra_df, obj_class_df)`
- **Parameters**:
    - `obj_spectra_df` : spectra of all objects (from part 3)
    - `obj_class_df`: object_id, classification (from part 1)
- **Returns**:
    - `df`: object id, photometry date, spectra date, difference between two, object type, number of spectra in Fritz


In [12]:
fritz_query.SpectraAnalysis.get_date_difference_df(obj_spectra_df, obj_class_df)

Processing Objects:   0%|          | 0/3 [00:00<?, ?it/s]

,obj_id,photometry_mjd,spectra_mjd,delta,type,data_length
0,ZTF17aaaobyl,60191.454317,58356.000000,1835.454317,Cataclysmic,1
1,ZTF18aabdajx,59623.328901,60390.366343,767.037442,Ia,1
2,ZTF18aatfxtd,58215.234942,59627.290012,1412.055069,Tidal Disruption Event,14
